In [ ]:
#| default_exp project

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

What kind of project a folder holds, and the steps that release it.

A folder is sniffed, not configured. `release_flow` names one of the flows in `FLOWS`, and
`default_steps` turns that name into the list of `Step` a pipeline runs. `Project` is the two of them
behind one object.

Nothing here runs a command or reaches a network. Every function takes a path and reads what is on
disk.

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass, field
from fastcore.all import Path

In [ ]:
#| export
@dataclass
class Step:
    "One command in a pipeline."
    id: str
    label: str
    cmd: str
    doc: str = ''
    needs: list = field(default_factory=list)   # environment keys the command reads
    argv: list = field(default_factory=list)    # spelled out, where `cmd` would not split correctly
    meta: dict = field(default_factory=dict)
    def dict(self): return {'id': self.id, 'label': self.label, 'cmd': self.cmd, 'doc': self.doc,
                            'needs': list(self.needs), 'meta': dict(self.meta)}

`Step` is one command, named by an `id` that is unique within its flow. `_with_app` finds its
insertion point by id, and a viewer addresses a running step by id.

`cmd` is the command as a person would type it, and it is what a viewer shows. `argv` is the same
command already split, for the cases where `shlex.split` would get it wrong or where the executable
has to be an absolute path. A step with an `argv` runs that; its `cmd` is then documentation.

`needs` names the environment keys the command reads. A pipeline asks the environment store for
those keys before the step starts, and reports the ones the store cannot produce.

`dict` is the wire form. It leaves `argv` out and copies `needs` and `meta`, so a client is told what
the step is rather than the resolved path this host will execute, and cannot edit the step by editing
what it was sent.

In [ ]:
s = Step('pypi', 'PyPI release', 'python -m twine upload dist/*',
         'Upload the built distributions to PyPI.', ['TWINE_USERNAME', 'TWINE_PASSWORD'])
s.dict()

In [ ]:
#| hide
test_eq(set(s.dict()), {'id', 'label', 'cmd', 'doc', 'needs', 'meta'})
s.dict()['needs'].append('SOMETHING_ELSE')
test_eq(s.needs, ['TWINE_USERNAME', 'TWINE_PASSWORD'])

In [ ]:
#| export
def _toml(root, name='pyproject.toml'):
    "One TOML file as a dict, or `{}` where it is missing or unreadable."
    import tomllib
    try:
        with open(Path(root)/name, 'rb') as f: return tomllib.load(f)
    except (OSError, ValueError): return {}

def nbdev_project(root):
    "An nbdev project: `settings.ini` for nbdev 2, or `[tool.nbdev]` in `pyproject.toml`."
    try:
        text = (Path(root)/'settings.ini').read_text(encoding='utf-8', errors='replace')
        if '[DEFAULT]' in text and any(k in text for k in ('nbs_path', 'lib_name', 'doc_path')):
            return True
    except OSError: pass
    data = _toml(root)
    return 'nbdev' in (data.get('tool') or {}) or 'nbdev' in ((data.get('project') or {}).get('entry-points') or {})

def rust_project(root):
    "A crate: something cargo can build, whether or not Python ever sees it."
    return (Path(root)/'Cargo.toml').exists()

def maturin_project(root):
    "A crate that is also a Python package: maturin builds the wheel, so the release is PyPI's."
    if not rust_project(root): return False
    data = _toml(root)
    backend = str((data.get('build-system') or {}).get('build-backend') or '')
    return backend.startswith('maturin') or 'maturin' in (data.get('tool') or {})

def fastship_project(root):
    "A Python package fastship releases: a `pyproject.toml`, and not an nbdev project."
    return (Path(root)/'pyproject.toml').exists() and not nbdev_project(root)

def app_project(root):
    "Whether the project packages itself into a desktop app, by the `setup_app.py` convention."
    root = Path(root)
    return (root/'tools'/'build_release.py').exists() and (root/'setup_app.py').exists()

Five questions about a folder, each answered by what is on disk.

`nbdev_project` takes either place nbdev keeps its configuration: a `settings.ini` holding
`[DEFAULT]` and one of nbdev's own paths, or `[tool.nbdev]` in `pyproject.toml`. `[DEFAULT]` on its
own is any ini file, so the paths are what decides.

`rust_project` is a `Cargo.toml`, whether or not Python ever sees the crate. `maturin_project` is a
crate whose `pyproject.toml` names maturin as the build backend or carries a `[tool.maturin]` table.

`fastship_project` is a `pyproject.toml` and no nbdev configuration. It is true of a maturin crate
too, and `release_flow` is where the order between them is settled.

`app_project` wants both `setup_app.py` and `tools/build_release.py`. One without the other is not a
desktop app.

`_toml` returns `{}` for a file that is missing and for one that does not parse. A half-written
`pyproject.toml` leaves a folder plain rather than raising.

In [ ]:
#| hide
tmp = TemporaryDirectory(); root = Path(tmp.name)
def mkproj(name, files=None):
    "A folder holding the files named, so a sniffer has something real to read."
    d = root/name; d.mkdir(parents=True, exist_ok=True)
    for p, text in (files or {}).items():
        f = d/p; f.parent.mkdir(parents=True, exist_ok=True); f.write_text(text)
    return d

In [ ]:
lib   = mkproj('a_notebook_lib', {'settings.ini': '[DEFAULT]\nlib_name = demo\nnbs_path = nbs\n'})
crate = mkproj('a_crate', {'Cargo.toml': '[package]\nname = "demo"\n'})
pkg   = mkproj('a_package', {'pyproject.toml': '[project]\nname = "demo"\n'})
[(p.name, nbdev_project(p), rust_project(p), fastship_project(p)) for p in (lib, crate, pkg)]

In [ ]:
#| hide
ep = mkproj('migrated', {'pyproject.toml': '[project.entry-points.nbdev]\ndemo = "demo._modidx:d"\n'})
assert nbdev_project(ep), 'the entry point is the other place nbdev is declared'
mat = mkproj('a_wheel_crate', {'Cargo.toml': '[package]\nname = "demo"\n',
                               'pyproject.toml': '[tool.maturin]\nmodule-name = "demo"\n'})
assert maturin_project(mat) and fastship_project(mat)
broken = mkproj('half_written', {'pyproject.toml': '[project\nname = '})
test_eq(_toml(broken), {})
assert not nbdev_project(broken) and fastship_project(broken)
assert not app_project(mkproj('half_an_app', {'setup_app.py': ''}))

In [ ]:
#| export
NBDEV_STEPS = [
    Step('prepare', 'nbdev_prepare', 'nbdev_prepare',
        'Export the notebooks, run the tests, clean the metadata, rebuild the README.'),
    Step('bump', 'bump version', 'nbdev_bump_version',
        'Raise the version in settings.ini and commit it.'),
    Step('gh', 'GitHub release', 'nbdev_release_gh',
        'Tag the version and create the GitHub release from CHANGELOG.', ['GITHUB_TOKEN']),
    Step('pypi', 'PyPI release', 'nbdev_pypi',
        'Build the wheel and upload it to PyPI.', ['TWINE_USERNAME', 'TWINE_PASSWORD']),
]

FASTSHIP_STEPS = [
    Step('test', 'test', 'python -m pytest -q', 'Run the test suite before anything is published.'),
    Step('changelog', 'changelog', 'ship-changelog',
        'Rewrite CHANGELOG.md from the issues closed since the last release.', ['GITHUB_TOKEN']),
    Step('gh', 'GitHub release', 'ship-gh --no_changelog --no_editor --yes',
        'Commit, push, tag, and create the GitHub release. The changelog step already wrote the notes.',
        ['GITHUB_TOKEN']),
    Step('pypi', 'PyPI release', 'ship-pypi',
        'Build the sdist and wheel, twine check them, then upload.',
        ['TWINE_USERNAME', 'TWINE_PASSWORD']),
    Step('bump', 'bump version', 'ship-bump',
        'Raise the version for the next cycle. fastship bumps after releasing, not before.'),
]

PLAIN_STEPS = [
    Step('test', 'test', 'python -m pytest -q', 'Run the test suite before anything is published.'),
    Step('build', 'build', 'python -m build', 'Build the sdist and wheel into dist/.'),
    Step('gh', 'GitHub release', 'gh release create --generate-notes',
        'Tag and publish a GitHub release.', ['GITHUB_TOKEN']),
    Step('pypi', 'PyPI release', 'python -m twine upload dist/*',
        'Upload the built distributions to PyPI.', ['TWINE_USERNAME', 'TWINE_PASSWORD']),
]

MATURIN_STEPS = [
    Step('test', 'test', 'cargo test', "Run the crate's own tests before anything is published."),
    Step('wheel', 'build the wheel', 'maturin build --release --out dist',
        'Build here first. A tag that fails to compile on the runners is a tag you have to undo.'),
    Step('ci', 'refresh the workflow', 'maturin generate-ci github -o .github/workflows/CI.yml',
        'maturin writes the matrix: linux, musllinux, macOS, Windows, and the sdist. Rewriting it '
        'from maturin keeps it right when your Python range or targets change.'),
    Step('changelog', 'changelog', 'ship-changelog',
        'Rewrite CHANGELOG.md from the issues closed since the last release.', ['GITHUB_TOKEN']),
    Step('release', 'tag and push', 'ship-release',
        'Tag the version from Cargo.toml and push it. The workflow builds every platform\'s wheel '
        'and publishes them, so nothing is uploaded from here.', ['GITHUB_TOKEN']),
]

CRATE_STEPS = [
    Step('test', 'test', 'cargo test', "Run the crate's tests before anything is published."),
    Step('clippy', 'clippy', 'cargo clippy --all-targets -- -D warnings',
        'The lints. A crate other people will read is worth the stricter pass.'),
    Step('package', 'package', 'cargo package',
        'Build the .crate the registry would receive, and verify it on its own.'),
    Step('publish', 'publish to crates.io', 'cargo publish',
        'Upload it. A version on crates.io cannot be replaced, only yanked.',
        ['CARGO_REGISTRY_TOKEN']),
]

APP_STEPS = [
    Step('app', 'desktop app', 'python tools/build_release.py',
        'Build the platform bundle: a .app and a DMG on macOS, a zipped folder on Windows.'),
    Step('assets', 'attach the app', 'python tools/build_release.py upload',
        'Upload what dist/ holds to the release this version just tagged.', ['GITHUB_TOKEN']),
]

#: Every flow, by the name `release_flow` answers with. Add one and `default_steps` can reach it.
FLOWS = {'nbdev': NBDEV_STEPS, 'fastship': FASTSHIP_STEPS, 'plain': PLAIN_STEPS,
         'maturin': MATURIN_STEPS, 'crate': CRATE_STEPS}

Five flows, each a list of `Step` in the order they run. `FLOWS` is data, so a kind pullup ships no
plan for is still reachable: pass your own dict to `default_steps` or to `Project`.

The nbdev flow bumps the version before it releases, and the fastship flow bumps after. Each tool
raises the version at a different point in its own cycle, and the step order is where that is written
down.

The maturin flow uploads nothing from here. `ship-release` pushes a tag and the workflow builds one
wheel per platform, because only the runners have the toolchains for all of them.

`APP_STEPS` is not in `FLOWS`. It is spliced into whichever flow a folder turns out to use.

In [ ]:
{name: [s.id for s in steps] for name, steps in FLOWS.items()}

In [ ]:
{s.id: s.needs for s in FASTSHIP_STEPS if s.needs}

In [ ]:
#| hide
app_ids = {s.id for s in APP_STEPS}
for name, steps in FLOWS.items():
    ids = [s.id for s in steps]
    test_eq(len(ids), len(set(ids)))
    test_eq(app_ids & set(ids), set())
    assert all(s.cmd and s.doc for s in steps), f'{name} has a step with nothing to run or say'

In [ ]:
#| export
def release_flow(root):
    """Which release flow this project uses.

    Rust is asked before fastship, because a maturin crate has a `pyproject.toml` too and the
    fastship flow would try to `ship-pypi` a wheel that only the runners can build.
    """
    if nbdev_project(root): return 'nbdev'
    if maturin_project(root): return 'maturin'
    if rust_project(root): return 'crate'
    return 'fastship' if fastship_project(root) else 'plain'

def _with_app(steps):
    "The app bundle and its upload, after the last step that publishes and before any that bumps."
    at = [i for i, s in enumerate(steps) if s.id in ('gh', 'pypi')]
    i = max(at) + 1 if at else len(steps)
    return [*steps[:i], *APP_STEPS, *steps[i:]]

def default_steps(root, flows=None):
    "The steps this kind of project is released by, with the desktop bundle where there is one."
    steps = list((flows or FLOWS)[release_flow(root)])
    return _with_app(steps) if app_project(root) else steps

`release_flow` asks the questions in an order that matters. nbdev first. Rust before fastship,
because a maturin crate has a `pyproject.toml` too and the fastship flow would `ship-pypi` a wheel
only the runners can build. A folder that answers nothing, including one that does not exist, is
`plain`.

`_with_app` splices the two app steps in after the last step that publishes, `gh` or `pypi`. `assets`
uploads to the release those steps just made, so it cannot run before them, and it has to run before
a bump that raises the version for the next cycle. A flow that publishes under neither id gets the
app steps at the end. The crate flow publishes with `cargo publish`, so no insertion point is found
and none is invented.

`default_steps` is the two of them together, and takes the same `flows` override.

In [ ]:
[(p.name, release_flow(p)) for p in (lib, crate, pkg, root/'no_such_folder')]

In [ ]:
app = mkproj('a_desktop_app', {'pyproject.toml': '[project]\nname = "demo"\n',
                               'setup_app.py': '', 'tools/build_release.py': ''})
[s.id for s in default_steps(app)]

`app` and `assets` sit after `pypi` and before `bump`, which is fastship's bump for the next cycle.

In [ ]:
#| hide
test_eq([s.id for s in default_steps(app)], ['test', 'changelog', 'gh', 'pypi', 'app', 'assets', 'bump'])
crate_app = mkproj('a_crate_app', {'Cargo.toml': '[package]\nname = "demo"\n',
                                   'setup_app.py': '', 'tools/build_release.py': ''})
ids = [s.id for s in default_steps(crate_app)]
test_eq(ids, ['test', 'clippy', 'package', 'publish', 'app', 'assets'])
mine = {'plain': [Step('only', 'the one step', 'true', 'nothing else runs')]}
test_eq([s.id for s in default_steps(root/'no_such_folder', flows=mine)], ['only'])

In [ ]:
#| export
class Project:
    "What kind of project one folder holds, and what releasing it takes."
    def __init__(self, root, flows=None):
        self.root, self.flows = Path(root).expanduser().resolve(), flows or FLOWS
    @property
    def kind(self): return release_flow(self.root)
    @property
    def packages_an_app(self): return app_project(self.root)
    def steps(self): return default_steps(self.root, self.flows)
    def dict(self):
        return {'root': str(self.root), 'kind': self.kind, 'app': self.packages_an_app,
                'steps': [s.dict() for s in self.steps()]}
    def __repr__(self): return f'Project({self.root.name}, {self.kind})'

`Project` answers the questions about one folder without a caller asking them one at a time. `root`
is expanded and resolved once, at construction. `kind` and `packages_an_app` read the disk on every
access, so a folder that gains a `Cargo.toml` changes kind under a `Project` that already exists.

`dict` is the whole answer as JSON-safe data: the resolved root as a string, the kind, whether there
is an app, and every step in wire form.

In [ ]:
p = Project(pkg)
d = p.dict()
p, d['kind'], d['app'], [s['id'] for s in d['steps']]

In [ ]:
#| hide
test_eq(d['root'], str(pkg.resolve()))
(pkg/'Cargo.toml').write_text('[package]\nname = "demo"\n')
test_eq(p.kind, 'crate')
(pkg/'Cargo.toml').unlink()
test_eq(p.kind, 'fastship')
test_eq(Project(root/'no_such_folder', flows=mine).steps(), mine['plain'])

In [ ]:
#| hide
tmp.cleanup()